# Telco Customer Churn — Exploratory Data Analysis

**Question:** which customers leave, and what do they have in common?

**Dataset:** IBM Telco Customer Churn (7,043 customers, 20 features + `Churn` target).

All heavy lifting (loading, cleaning) lives in `src/data.py` — this notebook is for looking, not for defining logic.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_raw, clean_data

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 40)

In [ ]:
raw = load_raw()
df = clean_data(raw)
print('Shape:', df.shape)
df.head()

## 1. Data quality check

In [ ]:
print('Missing values per column:')
print(df.isna().sum()[df.isna().sum() > 0] if df.isna().any().any() else 'None')
print()
print('Dtypes:')
df.dtypes

**Insight:** the only quality issue in the raw file was `TotalCharges` shipping as strings with 11 blank rows (tenure-0 customers). `clean_data()` handles this — after cleaning there are no missing values.

## 2. Target distribution — how imbalanced is churn?

In [ ]:
churn_rate = df['Churn'].mean()
print(f'Overall churn rate: {churn_rate:.1%}')

fig, ax = plt.subplots(figsize=(6, 4))
df['Churn'].value_counts().plot(kind='bar', ax=ax, color=['#2b8cbe', '#e34a33'])
ax.set_xticklabels(['Stayed', 'Churned'], rotation=0)
ax.set_title(f'Churn rate: {churn_rate:.1%}')
ax.set_ylabel('Customers')
plt.tight_layout(); plt.show()

**Insight:** ~26.5% churn — moderately imbalanced. A model that always predicts 'stayed' would get 73.5% accuracy and be useless. This is why we care more about **recall on the churn class** than accuracy (Phase 3).

## 3. Contract type — the single strongest predictor

In [ ]:
contract_churn = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
print(contract_churn.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
contract_churn.plot(kind='barh', ax=ax, color='#e34a33')
ax.set_xlabel('Churn rate'); ax.set_title('Churn by contract type')
plt.tight_layout(); plt.show()

**Insight:** month-to-month customers churn at **42.7%** — more than 15× the rate of two-year contracts (2.8%). Lock-in works. Any retention campaign should target month-to-month customers first.

## 4. Tenure — new customers are the risky ones

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', bins=30, ax=ax,
             palette={0: '#2b8cbe', 1: '#e34a33'})
ax.set_title('Tenure distribution, split by churn')
plt.tight_layout(); plt.show()

print(df.groupby('Churn')['tenure'].mean().round(1))

**Insight:** churners average **18 months** of tenure, stayers average **38 months**. The first year is the danger zone — most churn happens in months 1–12.

## 5. Internet service — fiber is oddly the biggest churn driver

In [ ]:
svc = df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False)
print(svc.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
svc.plot(kind='barh', ax=ax, color='#756bb1')
ax.set_xlabel('Churn rate'); ax.set_title('Churn by internet service')
plt.tight_layout(); plt.show()

**Insight:** fiber-optic customers churn at **41.9%** vs 19% for DSL and just 7.4% for no-internet customers. This is worth investigating — likely a mix of price sensitivity (fiber costs more) and service-quality complaints. It's a signal to the business, not just to the model.

## 6. Monthly charges — churners pay more

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True, ax=ax,
            palette={0: '#2b8cbe', 1: '#e34a33'})
ax.set_title('Monthly charges by churn status')
plt.tight_layout(); plt.show()

print(df.groupby('Churn')['MonthlyCharges'].mean().round(2))

**Insight:** churners pay $74/month on average, stayers pay $61. Higher bills correlate with churn — consistent with the fiber-optic finding above.

## 7. Numeric correlations

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='RdBu_r', center=0, fmt='.2f', ax=ax)
ax.set_title('Numeric feature correlations')
plt.tight_layout(); plt.show()

**Insight:** `tenure` and `TotalCharges` are strongly correlated (0.83) — expected, since total charges is roughly tenure × monthly. Tree-based models handle this fine; for the logistic-regression baseline we'll rely on scaling.

## Summary of EDA findings

| # | Finding | Implication |
|---|---------|-------------|
| 1 | Churn rate is 26.5% (imbalanced) | Optimize for recall / ROC-AUC, not accuracy |
| 2 | Month-to-month contracts churn at 42.7% vs 2.8% for 2-year | Contract type will be a top feature |
| 3 | Churners average 18-month tenure vs 38 for stayers | First year is the retention danger zone |
| 4 | Fiber-optic churns at 41.9% — highest of any service tier | Business signal: investigate fiber pricing/quality |
| 5 | Churners' monthly bills are ~20% higher | Price sensitivity matters |
| 6 | Data was clean apart from 11 blank `TotalCharges` (tenure-0) | Handled in `src/data.py` |
